In [16]:
import os
import pandas as pd


file_root_shapley = os.path.join(
    "..", "..", "data", "exact", "shapley_value_of_shoes.csv"
)
file_root_grabisch = os.path.join(
    "..", "..", "data", "exact", "grabisch_of_shoes.csv"
)
original = pd.read_csv(os.path.join(
    "..", "..", "data", "input", "shoes.csv"
), sep=";").set_index(
    "label",
    verify_integrity=True
)

shapley_only = pd.read_csv(
    file_root_shapley,
    sep=";",
    header=None,
    names=["value"]
)
shapley_only = shapley_only.set_index(
    original.index.values,
    verify_integrity=True
)

psi_1 = original.join(shapley_only)

r_1 = pd.read_csv(
    file_root_grabisch,
    sep=";",
    header=0
).set_index(
    original.index.values,
    verify_integrity=True
)
r_1.columns = original.index.values

In [17]:
import math
from cgt_perezsechi.manipulation.norm import normalize_psi, normalize_r


n = 4
psi_1_truncated = psi_1.copy()
psi_1_truncated["value"] = psi_1_truncated["value"].astype(float).apply(lambda number: math.floor(number * 10 ** n) / 10 ** n)

r_1_truncated = r_1.astype(float).apply(lambda row: row.apply(lambda number: math.floor(number * 10 ** n) / 10 ** n))

psi_2 = normalize_psi(psi_1)
psi_2_truncated = psi_2.copy()
psi_2_truncated["value"] = psi_2_truncated["value"].astype(float).apply(lambda number: math.floor(number * 10 ** n) / 10 ** n)

r_2 = normalize_r(r_1)
r_2_truncated = r_2.astype(float).apply(lambda row: row.apply(lambda number: math.floor(number * 10 ** n) / 10 ** n))

/home/secci/Localspace/UCM/thesis/thesis-python/.venv/lib/python3.10/site-packages/cgt_perezsechi/manipulation/norm.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  max_edge_width = r.applymap(lambda x: abs(x)).max().max()
/home/secci/Localspace/UCM/thesis/thesis-python/.venv/lib/python3.10/site-packages/cgt_perezsechi/manipulation/norm.py:9: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  r = r.copy().applymap(lambda x: x / max_edge_width)


In [18]:
from cgt_perezsechi.visualization.graph import draw

positive_alpha = 0
negative_alpha = 0
positive_beta = 0
negative_beta = 0
draw(
    psi=psi_2,
    r=r_2,
    positive_alpha=positive_alpha,
    negative_alpha=negative_alpha,
    positive_beta=positive_beta,
    negative_beta=negative_beta,
    output_path=os.path.join(
        "..", "..", "results", "shoes_network.png"
    )
)

/home/secci/Localspace/UCM/thesis/thesis-python/.venv/lib/python3.10/site-packages/cgt_perezsechi/visualization/graph.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  r = r.copy().applymap(filter_edge)
/home/secci/Localspace/UCM/thesis/thesis-python/.venv/lib/python3.10/site-packages/cgt_perezsechi/visualization/graph.py:55: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  adjacency = r.copy().applymap(lambda x: 1 if x != 0 else 0)
/home/secci/Localspace/UCM/thesis/thesis-python/.venv/lib/python3.10/site-packages/cgt_perezsechi/visualization/graph.py:240: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saving the graph to ../../results/shoes_network.png


In [19]:
def filter_to_adjacency_matrix(matrix, positive_beta, negative_beta):
    def filter_edge(x):
        if x > 0 and abs(x) > positive_beta:
            return 1
        elif x < 0 and abs(x) > negative_beta:
            return 1
        else:
            return 0
    
    return matrix.copy().applymap(filter_edge)

def filter_node_list(lst, positive_alpha, negative_alpha):
    def filter_node(x):
        if positive_alpha == 0 and negative_alpha == 0:
            return True
        elif x > 0 and abs(x) > positive_alpha:
            return True
        elif x < 0 and abs(x) > negative_alpha:
            return True
        else:
            return False
    return filter(filter_node, lst)

In [20]:
A = filter_to_adjacency_matrix(r_2, positive_beta, negative_beta).to_numpy()
M = r_2.copy().to_numpy()

/tmp/ipykernel_108563/2779306892.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return matrix.copy().applymap(filter_edge)


In [21]:
from cgt_perezsechi.modeling.cluster import duo_louvain

clusters = duo_louvain(A, M)
columns = r_2.columns
clusters = [[columns[x] for x in c] for c in clusters]

In [24]:
from cgt_perezsechi.visualization.graph import draw_clusters

node_pos = {
    "LS 1": (0, 4),
    "RS 1": (1, 4),
    "LS 2": (0, 3),
    "RS 2": (1, 3),
    "LS 3": (0, 2),
    "RS 3": (1, 2),
    "LS 4": (0, 1),
    "RS 4": (1, 1),
    "LS 5": (0, 0),
    "RS 5": (1, 0),
}

draw_clusters(
    psi=psi_2,
    r=r_2,
    clusters=clusters,
    node_pos=node_pos,
    positive_alpha=positive_alpha,
    negative_alpha=negative_alpha,
    positive_beta=positive_beta,
    negative_beta=negative_beta,
    output_path=os.path.join(
        "..", "..", "results", "shoes_clusters.png"
    )
)

/home/secci/Localspace/UCM/thesis/thesis-python/.venv/lib/python3.10/site-packages/cgt_perezsechi/visualization/graph.py:300: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  r = r.copy().applymap(filter_edge)
/home/secci/Localspace/UCM/thesis/thesis-python/.venv/lib/python3.10/site-packages/cgt_perezsechi/visualization/graph.py:301: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  adjacency = r.copy().applymap(lambda x: 1 if x != 0 else 0)
/home/secci/Localspace/UCM/thesis/thesis-python/.venv/lib/python3.10/site-packages/networkx/drawing/nx_pylab.py:457: UserWarning: *c* argument looks like a single numeric RGB or RGBA sequence, which should be avoided as value-mapping will have precedence in case its length matches with *x* & *y*.  Please use the *color* keyword-argument or provide a 2D array with a single row if you intend to specify the same RGB or RGBA value for all points.
  node_collection = ax.scatter(
/home/

Saving the graph to ../../results/shoes_clusters.png
